# Baseline Experiments

Compare text dataset distillation methods on classification.

For each method the notebook does the same five steps — visible in the helper below:

1. **Data** — `load_baseline_data(dataset_name, seed=SEED)`
2. **Selection** — `select_*(...)` (the method that makes this baseline different)
3. **Model** — `load_tokenizer(...)` + `load_sequence_classifier(...)`
4. **Training** — `train_text_classifier(model=..., tokenizer=..., ...)`
5. **Save** — `save_baseline_run(...)`

Selection runs once per dataset; training runs once per (dataset, model).

In [1]:
import sys
sys.path.insert(0, "/Users/ad-scherbakov/Documents/PythonProjects/momo_project/src")
from pathlib import Path

from text_distillation import (
    TimingTracker,
    load_baseline_data,
    save_baseline_run,
)
from text_distillation.analysis import collect_runs
from text_distillation.dilm import distill_dilm_official  # local paper DiLM self-generation pipeline
from text_distillation.distillation import (
    select_herding,
    select_kcenter_embeddings,
    select_kcenter_tfidf,
    select_random,
    select_stratified_random,
)
from text_distillation.model import (
    load_sequence_classifier,
    load_tokenizer,
    train_text_classifier,
)
from text_distillation.saving import create_run_dir
from text_distillation.vanilla_lm import distill_vanilla_lm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUNS_DIR = PROJECT_ROOT / "artifacts" / "runs"

In [2]:
# Shared across all baselines.
# Scope is currently restricted to SST-2 + bert-base-uncased so single-seed runs
# can finish quickly. To re-enable the full sweep, expand both lists.
DATASET_NAMES = ["sst2"]
MODEL_NAMES = ["bert-base-uncased"]
SEED = 42

# Used by selection methods (all except full_data).
K_PER_CLASS = 20

# Paper-faithful evaluator config — straight from DiLM-main/configs/test/coreset.yaml
# and dc.yaml, which all distillation/coreset rows in EXPECTED_METRICS.md use.
# Applied to every method except full_data (which keeps the default schedule).
PAPER_EVAL = dict(
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.5,
    scheduler_type="cosine",
    max_steps=200,
    train_batch_size=64,
    eval_batch_size=256,
    mixed_precision="bf16",  # paper uses bf16; CUDA-only — falls back to fp32 on CPU
    max_grad_norm=1.0,
)

# Encoder used by kcenter_cls / herding to embed the train pool.
EMBEDDING_MODEL_NAME = "bert-base-uncased"

In [3]:
def run_all_models(
    data, train_dataset, method_name, k_per_class, run_suffix_str, selection_tracker,
    **training_kwargs,
):
    """For one (dataset, method, distilled subset): train each model, save each run.

    `training_kwargs` is forwarded straight into `train_text_classifier`, so each
    section of the notebook can plug in its own schedule (e.g. PAPER_EVAL for
    distillation/coreset methods, default full-pass schedule for full_data).

    The model-loading step is inside the loop so each model gets a fresh classifier
    head sized for `data.num_labels`. Timings from `selection_tracker` (the same
    selection was used for every model on this dataset) are merged into each
    per-model tracker.
    """
    rows = []
    safe_dataset = data.dataset_info.name.replace("-", "_")
    for model_name in MODEL_NAMES:
        safe_model = model_name.replace("/", "_").replace("-", "_")
        run_dir = create_run_dir(RUNS_DIR, f"{method_name}_{safe_dataset}_{safe_model}_{run_suffix_str}")

        tracker = TimingTracker()
        tracker.merge(selection_tracker)

        # 3. Model — load tokenizer + classifier head sized for this dataset.
        tokenizer = load_tokenizer(model_name)
        model = load_sequence_classifier(
            model_name,
            num_labels=data.num_labels,
            label_names=data.label_names,
        )

        # 4. Training.
        _, metrics = train_text_classifier(
            model=model,
            tokenizer=tokenizer,
            train_dataset=train_dataset,
            eval_dataset=data.eval_dataset,
            output_dir=run_dir,
            text_columns=data.dataset_info.text_columns,
            metric_name=data.dataset_info.metric_name,
            seed=SEED,
            tracker=tracker,
            **training_kwargs,
        )

        # 5. Save config + metrics + distilled dataset.
        row = save_baseline_run(
            run_dir=run_dir,
            data=data,
            method_name=method_name,
            model_name=model_name,
            k_per_class=k_per_class,
            seed=SEED,
            train_dataset=train_dataset,
            metrics=metrics,
            tracker=tracker,
            project_root=PROJECT_ROOT,
        )
        rows.append(row)
        print(f"    [{model_name}] accuracy={metrics['accuracy']:.3f} f1_macro={metrics['f1_macro']:.3f}")
    return rows


all_results = {}

## 01 Full-Data Baseline

Train each model on the full train split per dataset — establishes the upper bound.

In [4]:
# method_name = "full_data"
# all_results[method_name] = []
# for dataset_name in DATASET_NAMES:
#     # 1. Data.
#     data = load_baseline_data(dataset_name, seed=SEED)
#     print(f"[{dataset_name}] {method_name} (no selection)")

#     # 2. Selection — no-op: train on the full pool.
#     train_dataset = data.train_pool
#     selection_tracker = TimingTracker()

#     all_results[method_name].extend(
#         run_all_models(data, train_dataset, method_name, None, "full", selection_tracker)
#     )

## 02 Random Coreset Baseline

Sample `K_PER_CLASS * num_labels` random examples (no class balancing).

In [5]:
method_name = "random"
all_results[method_name] = []
for dataset_name in DATASET_NAMES:
    # 1. Data.
    data = load_baseline_data(dataset_name, seed=SEED)
    k_total = K_PER_CLASS * data.num_labels
    print(f"[{dataset_name}] {method_name} k={k_total}")

    # 2. Selection — one random subset, reused across all models.
    selection_tracker = TimingTracker()
    with selection_tracker.measure("selection_sec"):
        train_dataset = select_random(data.train_pool, k=k_total, seed=SEED)

    all_results[method_name].extend(
        run_all_models(
            data, train_dataset, method_name, K_PER_CLASS, f"k{K_PER_CLASS}", selection_tracker,
            **PAPER_EVAL,
        )
    )

[sst2] random k=40


/Users/ad-scherbakov/miniforge3/envs/marketing/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Training 200 steps:   0%|          | 0/200 [00:00<?, ?it/s]

: 

## 03 Stratified Random Coreset Baseline

Sample `K_PER_CLASS` random examples per class — class-balanced version of 02.

In [ ]:
method_name = "stratified_random"
all_results[method_name] = []
for dataset_name in DATASET_NAMES:
    # 1. Data.
    data = load_baseline_data(dataset_name, seed=SEED)
    print(f"[{dataset_name}] {method_name} {K_PER_CLASS}/class")

    # 2. Selection — K_PER_CLASS per class, reused across all models.
    selection_tracker = TimingTracker()
    with selection_tracker.measure("selection_sec"):
        train_dataset = select_stratified_random(
            data.train_pool,
            k_per_class=K_PER_CLASS,
            label_column=data.dataset_info.label_column,
            seed=SEED,
        )

    all_results[method_name].extend(
        run_all_models(
            data, train_dataset, method_name, K_PER_CLASS, f"k{K_PER_CLASS}", selection_tracker,
            **PAPER_EVAL,
        )
    )

## 04 K-Center over TF-IDF Baseline

Greedy k-center per class on TF-IDF features — picks examples that cover the lexical space.

In [ ]:
method_name = "kcenter_tfidf"
all_results[method_name] = []
for dataset_name in DATASET_NAMES:
    # 1. Data.
    data = load_baseline_data(dataset_name, seed=SEED)
    print(f"[{dataset_name}] {method_name} {K_PER_CLASS}/class")

    # 2. Selection — k-center over TF-IDF, reused across all models.
    selection_tracker = TimingTracker()
    with selection_tracker.measure("selection_sec"):
        train_dataset = select_kcenter_tfidf(
            data.train_pool,
            text_columns=data.dataset_info.text_columns,
            label_column=data.dataset_info.label_column,
            k_per_class=K_PER_CLASS,
            seed=SEED,
        )

    all_results[method_name].extend(
        run_all_models(
            data, train_dataset, method_name, K_PER_CLASS, f"k{K_PER_CLASS}", selection_tracker,
            **PAPER_EVAL,
        )
    )

## 05 K-Center over Encoder Embeddings Baseline

Greedy k-center per class on pooled encoder embeddings — picks examples that cover the semantic space. Pooling is auto-selected per model (XLNet → last token).

In [ ]:
method_name = "kcenter_cls"
all_results[method_name] = []
for dataset_name in DATASET_NAMES:
    # 1. Data.
    data = load_baseline_data(dataset_name, seed=SEED)
    print(f"[{dataset_name}] {method_name} {K_PER_CLASS}/class (embedder={EMBEDDING_MODEL_NAME})")

    # 2. Selection — embed pool once with EMBEDDING_MODEL_NAME, then k-center per class.
    selection_tracker = TimingTracker()
    with selection_tracker.measure("selection_sec"):
        train_dataset = select_kcenter_embeddings(
            data.train_pool,
            text_columns=data.dataset_info.text_columns,
            label_column=data.dataset_info.label_column,
            k_per_class=K_PER_CLASS,
            model_name=EMBEDDING_MODEL_NAME,
            seed=SEED,
            tracker=selection_tracker,
        )

    all_results[method_name].extend(
        run_all_models(
            data, train_dataset, method_name, K_PER_CLASS, f"k{K_PER_CLASS}", selection_tracker,
            **PAPER_EVAL,
        )
    )

## 06 Herding Baseline

Greedy herding (Welling 2009) over pooled encoder embeddings — picks examples whose running mean best approximates the class mean.

In [ ]:
method_name = "herding"
all_results[method_name] = []
for dataset_name in DATASET_NAMES:
    # 1. Data.
    data = load_baseline_data(dataset_name, seed=SEED)
    print(f"[{dataset_name}] {method_name} {K_PER_CLASS}/class (embedder={EMBEDDING_MODEL_NAME})")

    # 2. Selection — embed pool once with EMBEDDING_MODEL_NAME, then herding per class.
    selection_tracker = TimingTracker()
    with selection_tracker.measure("selection_sec"):
        train_dataset = select_herding(
            data.train_pool,
            text_columns=data.dataset_info.text_columns,
            label_column=data.dataset_info.label_column,
            k_per_class=K_PER_CLASS,
            model_name=EMBEDDING_MODEL_NAME,
            seed=SEED,
            tracker=selection_tracker,
        )

    all_results[method_name].extend(
        run_all_models(
            data, train_dataset, method_name, K_PER_CLASS, f"k{K_PER_CLASS}", selection_tracker,
            **PAPER_EVAL,
        )
    )

## 07 Vanilla LM Baseline

Class-conditional GPT-2 fine-tuned on real text per class, then sampled. DiLM Table 1 baseline — synthesises new text instead of subsetting existing examples. **Only the 3 DiLM-paper datasets** (`sst2`, `qqp`, `mnli-m`); skipped for `ag_news`.

In [ ]:
DILM_PAPER_DATASETS = {"sst2", "qqp", "mnli-m"}
method_name = "vanilla_lm"
all_results[method_name] = []
for dataset_name in DATASET_NAMES:
    if dataset_name not in DILM_PAPER_DATASETS:
        print(f"[{dataset_name}] {method_name} skipped (not in DiLM paper scope)")
        continue
    # 1. Data.
    data = load_baseline_data(dataset_name, seed=SEED)
    print(f"[{dataset_name}] {method_name} {K_PER_CLASS}/class")

    # 2. Selection — train a class-conditional LM on real data, then sample.
    selection_tracker = TimingTracker()
    with selection_tracker.measure("selection_sec"):
        train_dataset = distill_vanilla_lm(
            data.train_pool,
            text_columns=data.dataset_info.text_columns,
            label_column=data.dataset_info.label_column,
            k_per_class=K_PER_CLASS,
            seed=SEED,
            tracker=selection_tracker,
        )

    all_results[method_name].extend(
        run_all_models(
            data, train_dataset, method_name, K_PER_CLASS, f"k{K_PER_CLASS}", selection_tracker,
            **PAPER_EVAL,
        )
    )

## 08 DiLM Baseline

Paper-faithful Maekawa et al. (NAACL Findings 2024) path: local port of the DiLM protocol, with no Hydra/MLflow and no runtime dependency on `DiLM-main/`. It trains LM for 80k steps, trains DC for 20k steps, generates 20 synthetic datasets, then loads one generated dataset for this notebook evaluation. This creates synthetic data locally; it does not use the pregenerated files under `DiLM-main/DiLM-synthetic-data/`.

In [ ]:
method_name = "dilm"
all_results[method_name] = []
for dataset_name in DATASET_NAMES:
    # 1. Data.
    data = load_baseline_data(dataset_name, seed=SEED)
    print(f"[{dataset_name}] {method_name} {K_PER_CLASS}/class (official 80k LM + 20k DC)")

    # 2. Selection — official DiLM pipeline: LM phase → DC phase → generate/evaluate datasets.
    selection_tracker = TimingTracker()
    with selection_tracker.measure("selection_sec"):
        train_dataset = distill_dilm_official(
            dataset_name=dataset_name,
            k_per_class=K_PER_CLASS,
            seed=SEED,
            dataset_index=0,
        )

    all_results[method_name].extend(
        run_all_models(
            data, train_dataset, method_name, K_PER_CLASS, f"k{K_PER_CLASS}", selection_tracker,
            **PAPER_EVAL,
        )
    )

## Aggregated results

Load every run from `artifacts/runs/` into a single DataFrame.

In [ ]:
df = collect_runs(RUNS_DIR)
df[["experiment_name", "method_name", "model_name", "dataset_name", "k_total",
    "accuracy", "f1_macro", "timings_selection_sec", "timings_training_sec"]].sort_values(
    ["dataset_name", "method_name", "model_name"]
)

## SST-2 overview

One row per (method, model). Older runs used aliases (`random_coreset`, `kcenter_cls_embeddings`) — those are normalized to the canonical names below. Methods are ordered from weakest baseline to most informed coreset, full-data on top.

In [ ]:
import pandas as pd

# Filter to SST-2 only, normalize older method-name aliases.
sst2 = df[df["dataset_name"] == "sst2"].copy()
sst2["method_name"] = sst2["method_name"].replace({
    "random_coreset": "random",
    "kcenter_cls_embeddings": "kcenter_cls",
})

# One row per (method, model). When the same pair has multiple runs (e.g. from
# different commits), keep the most recent one.
sst2 = (
    sst2.dropna(subset=["accuracy"])
        .drop_duplicates(subset=["method_name", "model_name"], keep="last")
)

# Order methods: full_data → random baselines → coreset methods.
method_order = ["full_data", "random", "stratified_random", "kcenter_tfidf", "kcenter_cls", "herding"]
sst2["method_name"] = pd.Categorical(sst2["method_name"], categories=method_order, ordered=True)
sst2 = sst2.sort_values(["method_name", "model_name"]).reset_index(drop=True)

# Compact view with only the columns that matter for comparison.
sst2_clean = sst2[[
    "method_name", "model_name", "k_total", "compression_ratio",
    "accuracy", "f1_macro",
    "timings_selection_sec", "timings_training_sec",
]].rename(columns={
    "method_name": "method",
    "model_name": "model",
    "k_total": "K",
    "compression_ratio": "compress",
    "timings_selection_sec": "sel_sec",
    "timings_training_sec": "train_sec",
})

# Persist for downstream analysis / sharing.
sst2_clean.to_csv(PROJECT_ROOT / "results228.csv", index=False)

# Render with formatting + heat-map on accuracy / f1.
sst2_clean.style.format({
    "K": "{:.0f}",
    "compress": "{:.1f}",
    "accuracy": "{:.3f}",
    "f1_macro": "{:.3f}",
    "sel_sec": "{:.1f}",
    "train_sec": "{:.1f}",
}).background_gradient(cmap="RdYlGn", subset=["accuracy", "f1_macro"], vmin=0.4, vmax=1.0)

### Accuracy by (method × model)

The headline view: rows = method, columns = classifier.

In [ ]:
sst2.pivot_table(
    index="method_name", columns="model_name", values="accuracy", observed=True
).round(3).style.background_gradient(cmap="RdYlGn", vmin=0.4, vmax=1.0).format("{:.3f}")